In [1]:
import pandas as pd
import tqdm, os
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../../..")

from utils.information_theoretic_calculation import InformationTheoreticCalculator
from utils.ipa_processing import syllabify_ipa
SEP = "."

In [2]:
df = pd.read_csv("../dataset_construction/datasets/processed_frmri.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[1])

In [3]:
df

,syllable1_start,syllable1_end,syllable2_start,syllable2_end,syllable1,syllable2,context,word,word_id,art_f,ac_f,subject
0,2.647,2.907,2.907,3.067,f_i,l_u,f_i,filou,2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
1,3.237,3.467,3.467,3.797,f_ʁ_i,p_u,f_ʁ_i,fripouille,5,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
2,3.467,3.797,3.797,4.017,p_u,j_ə,f_ʁ_i.p_u,fripouille,5,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
3,4.017,4.257,4.257,4.397,m_a,n_i,m_a,manipulent,6,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
4,4.257,4.397,4.397,4.797,n_i,p_y_l,m_a.n_i,manipulent,6,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
...,...,...,...,...,...,...,...,...,...,...,...,...
5167,29.683,30.123,30.123,30.313,s_i,z_ə,p_ʁ_e.s_i,précises,66,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4
5168,31.250,31.440,31.440,31.890,p_ʁ_e,s_i,p_ʁ_e,précises,72,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4
5169,31.440,31.890,31.890,32.090,s_i,z_ə,p_ʁ_e.s_i,précises,72,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4
5170,32.981,33.211,33.211,33.671,p_ʁ_e,s_i,p_ʁ_e,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4


In [4]:
with open("../../../phone_based/pronunciation_counts/pronunciation_counts_fr.pkl", "rb") as f:
    pronunciation_counts = pickle.load(f)
pronunciation_counts = {syllabify_ipa(k, lang="fr"): v for k, v in pronunciation_counts.items() if "*" not in k and k != ""}

In [5]:
bigrams = set()
for syllable1, syllable2 in tqdm.tqdm(zip(df["syllable1"], df["syllable2"]), total=len(df)):
    bigrams.add(syllable1+SEP+syllable2)

all_phones = set()
for pronunciation in tqdm.tqdm(pronunciation_counts):

    for phone in pronunciation.split(SEP):
        all_phones.add(phone)

monogram_counts = {}
for pronunciation, count in tqdm.tqdm(pronunciation_counts.items()):
    syllables = pronunciation.split(SEP)
    for syllable in syllables:
        if syllable not in monogram_counts:
            monogram_counts[syllable] = 0
        monogram_counts[syllable] += count

  0%|          | 0/7193272 [00:00<?, ?it/s]

100%|██████████| 7193272/7193272 [00:05<00:00, 1257962.01it/s]


In [6]:
calculator = InformationTheoreticCalculator(
    pronunciation_counts=pronunciation_counts,
    phone_counts=monogram_counts,
    sep=SEP
)

100%|██████████| 7193272/7193272 [00:30<00:00, 233094.00it/s]


In [7]:
df["bigram_surprisal"] = df.apply(lambda x: calculator.get_contextual_surprisal(SEP.join(x["context"].split(SEP)[:-1]), x["syllable1"] + SEP + x["syllable2"]), axis=1)
df["syllable2_pos_entropy"] = df.apply(lambda x: calculator.get_target_pos_entropy(x["context"]), axis=1)

In [8]:
df

,syllable1_start,syllable1_end,syllable2_start,syllable2_end,syllable1,syllable2,context,word,word_id,art_f,ac_f,subject,bigram_surprisal,syllable2_pos_entropy
0,2.647,2.907,2.907,3.067,f_i,l_u,f_i,filou,2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6,13.086142,3.013671
1,3.237,3.467,3.467,3.797,f_ʁ_i,p_u,f_ʁ_i,fripouille,5,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6,18.029568,3.801717
2,3.467,3.797,3.797,4.017,p_u,j_ə,f_ʁ_i.p_u,fripouille,5,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6,NaN,1.073543
3,4.017,4.257,4.257,4.397,m_a,n_i,m_a,manipulent,6,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6,8.787051,4.078539
4,4.257,4.397,4.397,4.797,n_i,p_y_l,m_a.n_i,manipulent,6,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6,8.561736,1.713917
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5167,29.683,30.123,30.123,30.313,s_i,z_ə,p_ʁ_e.s_i,précises,66,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4,NaN,1.725383
5168,31.250,31.440,31.440,31.890,p_ʁ_e,s_i,p_ʁ_e,précises,72,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4,8.502885,3.632211
5169,31.440,31.890,31.890,32.090,s_i,z_ə,p_ʁ_e.s_i,précises,72,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4,NaN,1.725383
5170,32.981,33.211,33.211,33.671,p_ʁ_e,s_i,p_ʁ_e,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4,8.502885,3.632211
